In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
reviews = session.table(
    "VP_ANALYTICS_PROJECT.TEAM_7.REVIEW"
)

reviews.count()

In [ ]:
#Use Chris' qsr list to get a list of relevant businesses to our analysis
import pandas as pd

qsr_option_b = pd.read_csv('qsr_option_b.csv')
print(qsr_option_b.head())
print(qsr_option_b.columns)

In [ ]:
#selecting qsr businesses with no grouping or aggregation

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
session.use_database("VP_ANALYTICS_PROJECT")
session.use_schema("TEAM_7")

from snowflake.snowpark import functions as F

# Get the list of QSR business IDs
qsr_ids = qsr_option_b["business_id"].tolist()

# Get the REVIEW table
reviews = session.table(
    "VP_ANALYTICS_PROJECT.TEAM_7.REVIEW"
)

# Filter to QSR businesses and aggregate in Snowflake
review_qsr = (
    reviews
    .filter(F.col("BUSINESS_ID").isin(qsr_ids))
).to_pandas()



In [ ]:
session.write_pandas(
    review_qsr,
    "REVIEWS_QSR_OPTION_B",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
#slice and dice data by review score - 1 star
onestar_reviews = review_qsr[review_qsr['STARS']==1]
onestar_reviews.head()

In [ ]:
#slice and dice data by review score - 2 star
twostar_reviews = review_qsr[review_qsr['STARS']==2]
twostar_reviews.head()


In [ ]:
#three star reviews
threestar_reviews = review_qsr[review_qsr['STARS']==3]
threestar_reviews.head()

In [ ]:
#four star reviews
fourstar_reviews = review_qsr[review_qsr['STARS']==4]
fourstar_reviews.head()

In [ ]:
#five star reviews
fivestar_reviews = review_qsr[review_qsr['STARS']==5]
fivestar_reviews.head()

In [ ]:
print('One star review total:', onestar_reviews['REVIEW_ID'].count())
print('Two star review total:', twostar_reviews['REVIEW_ID'].count())
print('Three star review total:', threestar_reviews['REVIEW_ID'].count())
print('Four star review total:', fourstar_reviews['REVIEW_ID'].count())
print('Five star review total:', fivestar_reviews['REVIEW_ID'].count())

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

sia = SentimentIntensityAnalyzer()

chunk_size = 35000
results = []

for start in range(0, len(review_qsr), chunk_size):

    chunk = review_qsr.iloc[start:start + chunk_size].copy()

    print(f"Processing {start:,} to {min(start + chunk_size, len(review_qsr)):,}")

    chunk["sentiment_score"] = (
        chunk["REVIEW_TEXT"]
        .fillna("")
        .astype(str)
        .apply(lambda x: sia.polarity_scores(x)["compound"])
    )

    results.append(
        chunk[["REVIEW_ID", "USER_ID","BUSINESS_ID", "STARS", "FUNNY", "USEFUL","COOL", "REVIEW_DATE","sentiment_score"]]
    )



In [ ]:
review_sentiment = pd.concat(results, ignore_index=True)
review_sentiment.head()

In [ ]:
session.write_pandas(
    review_sentiment,
    "REVIEW_SENTIMENT",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
import pandas as pd

review_sentiment = session.table(
    "VP_ANALYTICS_PROJECT.TEAM_7.REVIEW_SENTIMENT"
).to_pandas()

In [ ]:
#aggregate reviews by business id now

sentiment_agg = (
    review_sentiment
    .groupby("BUSINESS_ID")
    .agg(
        avg_stars=("STARS", "mean"),
        avg_sentiment=("sentiment_score", "mean"),
        funny_count=("FUNNY", "count"),
        useful_count=("USEFUL","count"),
        cool_count=("COOL", "count")
    )
    .reset_index()
)
sentiment_agg.head()

In [ ]:
session.write_pandas(
    sentiment_agg,
    "SENTIMENT_AGGREGATED_BUS_ID",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
#Do some aggregations by geography

# Get the REVIEW table
business = session.table(
    "VP_ANALYTICS_PROJECT.TEAM_7.BUSINESS"
).to_pandas()

sentiment_bus_agg = session.table("VP_ANALYTICS_PROJECT.TEAM_7.SENTIMENT_AGGREGATED_BUS_ID").to_pandas()

# Filter to QSR businesses and aggregate in Snowflake
sentiment_geog_agg = sentiment_bus_agg.merge(
    business[["BUSINESS_ID", "STATE", "CITY"]],
    on="BUSINESS_ID",
    how="inner"
)


In [ ]:
sentiment_geog_agg.head()

In [ ]:
sentiment_geog_agg = sentiment_geog_agg.groupby(['STATE','CITY']).agg(
        avg_stars=("avg_stars", "mean"),
        funny_count=("funny_count", "count"),
        useful_count=("useful_count","count"),
        cool_count=("cool_count", "count"),
        avg_sentiment=("avg_sentiment", "mean")).reset_index()


sentiment_geog_agg.head()

In [ ]:
session.write_pandas(
    sentiment_geog_agg,
    "SENTIMENT_AGGREGATED_GEOG",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
#Do some analysis of sentiment over time

# Get the REVIEW table
review_sentiment = session.table(
    "VP_ANALYTICS_PROJECT.TEAM_7.REVIEW_SENTIMENT"
).to_pandas()

#convert the date string to a datetime
review_sentiment["REVIEW_DATE"] = pd.to_datetime(review_sentiment["REVIEW_DATE"], dayfirst=True)

review_sentiment.head()



In [ ]:
#Let's aggregate by date

review_sentiment["REVIEW_DATE"] = review_sentiment["REVIEW_DATE"].dt.date

review_agg_time = review_sentiment.groupby('REVIEW_DATE').agg(
        avg_stars=("STARS", "mean"),
        funny_count=("FUNNY", "count"),
        useful_count=("USEFUL","count"),
        cool_count=("COOL", "count"),
        avg_sentiment=("sentiment_score", "mean")).reset_index()



In [ ]:
session.write_pandas(
    review_agg_time,
    "SENTIMENT_AGGREGATED_TIME",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
#selecting with qsr business and aggregating reviews

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
session.use_database("VP_ANALYTICS_PROJECT")
session.use_schema("TEAM_7")

from snowflake.snowpark import functions as F

# Get the list of QSR business IDs
qsr_ids = qsr_option_b["business_id"].tolist()

# Get the REVIEW table
reviews = session.table(
    "VP_ANALYTICS_PROJECT.TEAM_7.REVIEW"
)

# Filter to QSR businesses and aggregate in Snowflake
review_agg = (
    reviews
    .filter(F.col("BUSINESS_ID").isin(qsr_ids))
    .group_by("BUSINESS_ID")
    .agg(
        F.avg("STARS").alias("AVG_STARS"),
        F.count("STARS").alias("REVIEW_COUNT"),
        F.array_agg("REVIEW_TEXT").alias("REVIEWS")
    )
)

# Only bring the aggregated result into Pandas
reviews_df = review_agg.to_pandas()

# Make column names lowercase
reviews_df.columns = reviews_df.columns.str.lower()

reviews_df.head()

In [ ]:
type(reviews_df)

In [ ]:
reviews_df.shape

In [ ]:
#try aggregate by business and drill down

review_agg_bus = (
    reviews.groupby(["business_id", "review_date"])
           .agg(
               review_count=("review_date", "count"),
               review_tex=("stars", "mean")
           )
           .reset_index()

)

review_agg_bus

In [ ]:
#This will set up some sentiment analysis for the reviews

# Transform data to lowercase.
reviews_df['reviews'] = reviews_df['reviews'].apply(lambda x: " ".join(x.lower() for x in x.split()))
reviews_df.head()

In [ ]:
#pull out the punctuation marks
reviews_df["reviews"] = reviews_df["reviews"].str.replace(
    r"[^\w\s]", "", regex=True
)

In [ ]:
#check for duplicates

reviews_df['reviews'].duplicated().sum()

In [ ]:
#drop any duplicates
reviews = reviews.drop_duplicates(subset=['review_text'])
reviews.head()

In [ ]:
review_qsr_pd = review_qsr.select(
    "BUSINESS_ID",
    "REVIEW_TEXT",
    "STARS"
).to_pandas()

In [ ]:
review_qsr.count()

In [ ]:
review_qsr_pd.head()

In [ ]:
review_qsr_agg=review_qsr_pd.groupby('BUSINESS_ID')

In [ ]:
review_qsr_agg.shape()

In [ ]:

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()
print("VADER loaded successfully")

In [ ]:
#do sentiment analysis for reviews
reviews_df["sentiment_scores"] = reviews_df["reviews"].apply(sid.polarity_scores)

In [ ]:
reviews_df["reviews"].isna().sum()

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

review_df["sentiment_score"] = review_df["reviews"].fillna("").apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

In [ ]:
type(reviews_df)

In [ ]:
SELECT AI_SENTIMENT('This food was absolutely fantastic');

In [ ]:


review_df.head()

In [ ]:
from snowflake.snowpark import functions as F

reviews_sentiment = (
    review_qsr
    .select(
        "BUSINESS_ID",
        "REVIEW_TEXT",
        "STARS",
        F.call_function(
            "AI_SENTIMENT",
            F.col("REVIEW_TEXT")
        ).alias("SENTIMENT")
    )
)

In [ ]:

reviews_sentiment.head()

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

review_qsr["sentiment_score"] = review_qsr["review_text"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)